In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import tkinter as tk
from tkinter import filedialog

# 1. 選擇並讀取工具輸出的 S 參數 CSV
root = tk.Tk()
root.withdraw()
csv_path = filedialog.askopenfilename(title="選擇 CSV 檔案",
                                      filetypes=[("CSV files", "*.csv"), ("All files", "*.*")])
df = pd.read_csv(csv_path)

freq_col = "Frequency_GHz" if "Frequency_GHz" in df.columns else df.columns[0]
freqs = df[freq_col].values

# 自動偵測所有 S 參數 (S11, S21, ... )
s_param_names = sorted({c.split("_")[0] for c in df.columns
                        if c.startswith("S") and "_" in c and c.split("_")[0][1:].isdigit()})
print(f"✅ 偵測到的 S 參數有 ({len(s_param_names)} 個): {s_param_names}")

# 想只畫部分參數時改這裡，例如 ['S11', 'S21']
target_plots = s_param_names

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for s in target_plots:
    if f"{s}_dB" in df.columns:
        ax1.plot(freqs, df[f"{s}_dB"], label=s, linewidth=2)
    if f"{s}_phase" in df.columns:
        ax2.plot(freqs, df[f"{s}_phase"], label=s, linewidth=2, linestyle='--')

ax1.set_title("Magnitude Response (dB)", fontsize=14, fontweight='bold')
ax1.set_xlabel("Frequency (GHz)"); ax1.set_ylabel("Magnitude (dB)")
ax1.grid(True, which='both', linestyle=':', alpha=0.7); ax1.legend(loc='best')

ax2.set_title("Phase Response (Degree)", fontsize=14, fontweight='bold')
ax2.set_xlabel("Frequency (GHz)"); ax2.set_ylabel("Phase (deg)")
ax2.grid(True, which='both', linestyle=':', alpha=0.7); ax2.legend(loc='best')
ax2.set_yticks(np.arange(-180, 181, 45))

plt.tight_layout()
plt.show()


In [ ]:
# 把每個 S 參數拆成獨立變數，方便後續分析 (例如 S21_dB、S21_linear)
for s in target_plots:
    globals()[f"{s}_dB"] = df[f"{s}_dB"]
    globals()[f"{s}_real"] = df[f"{s}_real"]
    globals()[f"{s}_imag"] = df[f"{s}_imag"]
    globals()[f"{s}_linear"] = 10 ** (df[f"{s}_dB"] / 20)   # 振幅 (dB -> 線性)；舊版誤用 np.exp
    globals()[f"{s}_phase"] = df[f"{s}_phase"]


In [ ]:
plt.figure(figsize=(15, 3))
plt.plot(freqs, S11_dB, label="S11")
plt.plot(freqs, S21_dB, label="S21")
plt.legend()
plt.xlabel("freq (GHz)")
plt.show()
